# UTCI Sulawesi 1996–2025 — STEP 2
## Kelvin → °C + Daily Maximum UTCI + Mask Sulawesi + QC

Notebook ini dijalankan **setelah STEP 1 QC UTCI RAW dinyatakan LULUS**.

### Input
- UTCI hourly harian:
  `D:\MERLI\data\UTCI\1996-2025`
- QC STEP 1:
  `D:\MERLI\data\UTCI_Hasil\01_qc_utci_raw_files_1996_2025.csv`
- Mask Sulawesi dari pengolahan Tmax:
  `D:\MERLI\data\TMAX_Hasil\00_sulawesi_mask_native_grid.nc`

### Output
`D:\MERLI\data\UTCI_Hasil\03_utci_dailymax_1996_2025_C_sulawesi.nc`

### Alur
`UTCI hourly → daily maximum → Kelvin ke °C → crop/mask Sulawesi → satu NetCDF harian → QC`

**Catatan waktu:** daily maximum dihitung dari 24 timestep UTC (00–23 UTC) per file, konsisten dengan struktur file UTCI dan basis harian Tmax yang sebelumnya digunakan.

In [1]:
# =========================================================
# 0. IMPORT
# =========================================================

from pathlib import Path
import gc
import calendar

import numpy as np
import pandas as pd
import xarray as xr

from netCDF4 import Dataset

print("xarray :", xr.__version__)
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

xarray : 2025.10.1
pandas : 2.3.3
numpy  : 2.3.5


In [2]:
# =========================================================
# 1. KONFIGURASI
# =========================================================

UTCI_DIR = Path(r"D:\MERLI\data\UTCI\1996-2025")
UTCI_RESULT_DIR = Path(r"D:\MERLI\data\UTCI_Hasil")

TMAX_MASK_FILE = Path(
    r"D:\MERLI\data\TMAX_Hasil\00_sulawesi_mask_native_grid.nc"
)

QC_STEP1_FILE = UTCI_RESULT_DIR / "01_qc_utci_raw_files_1996_2025.csv"

OUTPUT_FILE = (
    UTCI_RESULT_DIR
    / "03_utci_dailymax_1996_2025_C_sulawesi.nc"
)

QC_OUTPUT_FILE = (
    UTCI_RESULT_DIR
    / "04_qc_utci_dailymax_1996_2025.csv"
)

START_YEAR = 1996
END_YEAR = 2025

# Berdasarkan file contoh + QC STEP 1:
# data UTCI disimpan dalam Kelvin.
INPUT_IS_KELVIN = True

FILL_VALUE = np.float32(-9999.0)

UTCI_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not UTCI_DIR.exists():
    raise FileNotFoundError(
        f"Folder UTCI tidak ditemukan:\n{UTCI_DIR}"
    )

if not TMAX_MASK_FILE.exists():
    raise FileNotFoundError(
        "Mask Sulawesi dari Tmax tidak ditemukan:\n"
        f"{TMAX_MASK_FILE}"
    )

print("UTCI input :", UTCI_DIR)
print("Mask       :", TMAX_MASK_FILE)
print("Output     :", OUTPUT_FILE)

UTCI input : D:\MERLI\data\UTCI\1996-2025
Mask       : D:\MERLI\data\TMAX_Hasil\00_sulawesi_mask_native_grid.nc
Output     : D:\MERLI\data\UTCI_Hasil\03_utci_dailymax_1996_2025_C_sulawesi.nc


## 2. Baca daftar file dari QC STEP 1

Kalau CSV QC tersedia, notebook memakai daftar file dari sana karena tanggal dan path-nya sudah diverifikasi. Jika tidak ada, notebook mencari file `.nc` secara rekursif.

In [3]:
# =========================================================
# 2. DAFTAR FILE UTCI
# =========================================================

if QC_STEP1_FILE.exists():

    qc1 = pd.read_csv(
        QC_STEP1_FILE,
        parse_dates=[
            "date",
            "time_start",
            "time_end"
        ]
    )

    if "path" not in qc1.columns:
        raise ValueError(
            "Kolom 'path' tidak ada di CSV QC STEP 1."
        )

    qc1 = (
        qc1
        .sort_values("date")
        .reset_index(drop=True)
    )

    input_files = [
        Path(p)
        for p in qc1["path"]
    ]

    print(
        "Memakai daftar file dari QC STEP 1."
    )

else:

    input_files = sorted(
        [
            p for p in UTCI_DIR.rglob("*")
            if p.is_file()
            and p.suffix.lower() == ".nc"
        ]
    )

    print(
        "QC STEP 1 CSV tidak ditemukan; "
        "mencari file langsung dari folder."
    )


expected_dates = pd.date_range(
    f"{START_YEAR}-01-01",
    f"{END_YEAR}-12-31",
    freq="D"
)

print("Jumlah file :", len(input_files))
print("Hari ideal  :", len(expected_dates))

if len(input_files) != len(expected_dates):
    raise ValueError(
        "Jumlah file UTCI tidak sama dengan jumlah hari ideal. "
        "STEP 1 seharusnya sudah LULUS sebelum menjalankan STEP 2."
    )

missing_paths = [
    str(p)
    for p in input_files
    if not p.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Ada path file dari QC yang sudah tidak tersedia. "
        f"Contoh:\n{missing_paths[:5]}"
    )

print("Daftar file: LULUS")

Memakai daftar file dari QC STEP 1.
Jumlah file : 10958
Hari ideal  : 10958
Daftar file: LULUS


## 3. Baca mask Sulawesi dari hasil Tmax

Dengan memakai mask Tmax yang sama, wilayah/grid UTCI dan Tmax dibuat konsisten untuk analisis gabungan berikutnya.

In [4]:
# =========================================================
# 3. LOAD MASK SULAWESI TMAX
# =========================================================

with xr.open_dataset(
    TMAX_MASK_FILE,
    decode_timedelta=False
) as mask_ds:

    if "sulawesi_mask" not in mask_ds:
        raise ValueError(
            "Variabel 'sulawesi_mask' tidak ditemukan."
        )

    mask_crop = np.asarray(
        mask_ds["sulawesi_mask"].values,
        dtype=bool
    )

    mask_lat = np.asarray(
        mask_ds["latitude"].values,
        dtype=float
    )

    mask_lon = np.asarray(
        mask_ds["longitude"].values,
        dtype=float
    )

print("Mask shape :", mask_crop.shape)
print("Grid darat :", int(mask_crop.sum()))
print(
    "Lat        :",
    mask_lat.min(),
    "s.d.",
    mask_lat.max()
)
print(
    "Lon        :",
    mask_lon.min(),
    "s.d.",
    mask_lon.max()
)

if int(mask_crop.sum()) == 0:
    raise ValueError(
        "Mask Sulawesi kosong."
    )

Mask shape : (44, 32)
Grid darat : 242
Lat        : -6.25 s.d. 4.5
Lon        : 119.0 s.d. 126.75


## 4. Inspeksi satu file dan cocokkan grid

Notebook mencari indeks latitude/longitude UTCI yang sesuai dengan grid crop Sulawesi dari Tmax.

In [5]:
# =========================================================
# 4. CEK FILE CONTOH + MATCH GRID
# =========================================================

sample_file = input_files[0]

with xr.open_dataset(
    sample_file,
    decode_times=True,
    decode_timedelta=False
) as ds_sample:

    if "utci" not in ds_sample.data_vars:
        raise ValueError(
            f"{sample_file.name}: variabel 'utci' tidak ditemukan."
        )

    da = ds_sample["utci"]

    # Standardisasi koordinat
    rename_map = {}

    if "lat" in da.coords:
        rename_map["lat"] = "latitude"

    if "lon" in da.coords:
        rename_map["lon"] = "longitude"

    if rename_map:
        da = da.rename(rename_map)

    da = da.transpose(
        "time",
        "latitude",
        "longitude"
    )

    full_lat = np.asarray(
        da["latitude"].values,
        dtype=float
    )

    full_lon = np.asarray(
        da["longitude"].values,
        dtype=float
    )

    sample_times = pd.DatetimeIndex(
        da["time"].values
    )

    raw_min = float(
        da.min(skipna=True).values
    )

    raw_max = float(
        da.max(skipna=True).values
    )


def find_matching_indices(full_coord, target_coord, tol=1e-6):
    idx = []

    for value in target_coord:
        hits = np.where(
            np.isclose(
                full_coord,
                value,
                atol=tol,
                rtol=0
            )
        )[0]

        if len(hits) != 1:
            raise ValueError(
                f"Koordinat {value} tidak cocok unik "
                "dengan grid UTCI."
            )

        idx.append(
            int(hits[0])
        )

    return np.asarray(
        idx,
        dtype=int
    )


lat_idx = find_matching_indices(
    full_lat,
    mask_lat
)

lon_idx = find_matching_indices(
    full_lon,
    mask_lon
)

print("File contoh :", sample_file.name)
print("Shape UTCI  :", da.shape)
print("n_time      :", len(sample_times))
print("Jam awal    :", sample_times.min())
print("Jam akhir   :", sample_times.max())
print("Raw min/max :", raw_min, "/", raw_max)
print("Crop target :", (len(lat_idx), len(lon_idx)))
print("Mask shape  :", mask_crop.shape)

if len(sample_times) != 24:
    raise ValueError(
        "File contoh tidak memiliki 24 timestep."
    )

if mask_crop.shape != (
    len(lat_idx),
    len(lon_idx)
):
    raise ValueError(
        "Shape mask tidak cocok dengan crop UTCI."
    )

print("GRID UTCI ↔ MASK TMAX: LULUS")

File contoh : ECMWF_utci_19960101_v1.1_con.area-subset.7.127.-7.116.nc
Shape UTCI  : (24, 57, 45)
n_time      : 24
Jam awal    : 1996-01-01 00:00:00
Jam akhir   : 1996-01-01 23:00:00
Raw min/max : 285.3293151855469 / 311.45562744140625
Crop target : (44, 32)
Mask shape  : (44, 32)
GRID UTCI ↔ MASK TMAX: LULUS


## 5. Tes Daily Maximum satu hari

Belum menulis file 30 tahun. Cell ini hanya memastikan daily maximum dan konversi °C masuk akal.

In [6]:
# =========================================================
# 5. TES DAILY MAX SATU FILE
# =========================================================

with xr.open_dataset(
    sample_file,
    decode_times=True,
    decode_timedelta=False
) as ds:

    da = ds["utci"]

    rename_map = {}

    if "lat" in da.coords:
        rename_map["lat"] = "latitude"

    if "lon" in da.coords:
        rename_map["lon"] = "longitude"

    if rename_map:
        da = da.rename(rename_map)

    da = da.transpose(
        "time",
        "latitude",
        "longitude"
    )

    da_crop = da.isel(
        latitude=lat_idx,
        longitude=lon_idx
    )

    # Daily maximum dari 24 jam
    daily_max = np.asarray(
        da_crop.max(
            dim="time",
            skipna=True
        ).values,
        dtype="float32"
    )

    # Kelvin → Celsius
    if INPUT_IS_KELVIN:
        daily_max -= np.float32(273.15)

    # Mask daratan Sulawesi
    daily_max[
        ~mask_crop
    ] = np.nan


print(
    "Daily max minimum:",
    float(np.nanmin(daily_max)),
    "°C"
)

print(
    "Daily max maksimum:",
    float(np.nanmax(daily_max)),
    "°C"
)

print(
    "Grid valid:",
    int(np.isfinite(daily_max).sum())
)

if int(np.isfinite(daily_max).sum()) != int(mask_crop.sum()):
    raise ValueError(
        "Jumlah grid valid daily max tidak sama dengan mask Sulawesi."
    )

print("TES DAILY MAX SATU HARI: LULUS")

del daily_max
gc.collect()

Daily max minimum: 26.398681640625 °C
Daily max maksimum: 38.305633544921875 °C
Grid valid: 242
TES DAILY MAX SATU HARI: LULUS


88

## 6. Proses seluruh 1996–2025 secara streaming

Setiap file:
1. dibuka;
2. dicek 24 jam;
3. diambil maksimum hariannya;
4. dikonversi K → °C;
5. dicrop/mask Sulawesi;
6. langsung ditulis ke NetCDF output.

Jadi 10.958 file **tidak ditumpuk ke RAM**.

In [7]:
# =========================================================
# 6. STREAMING SEMUA HARI KE SATU NETCDF
# =========================================================

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()


with Dataset(
    OUTPUT_FILE,
    "w",
    format="NETCDF4"
) as out:

    # -------------------------
    # DIMENSI
    # -------------------------

    out.createDimension(
        "time",
        len(input_files)
    )

    out.createDimension(
        "latitude",
        len(mask_lat)
    )

    out.createDimension(
        "longitude",
        len(mask_lon)
    )


    # -------------------------
    # KOORDINAT
    # -------------------------

    time_var = out.createVariable(
        "time",
        "f8",
        ("time",)
    )

    lat_var = out.createVariable(
        "latitude",
        "f4",
        ("latitude",)
    )

    lon_var = out.createVariable(
        "longitude",
        "f4",
        ("longitude",)
    )


    # -------------------------
    # UTCI DAILY MAX
    # -------------------------

    utci_var = out.createVariable(
        "utci_max",
        "f4",
        (
            "time",
            "latitude",
            "longitude"
        ),
        zlib=True,
        complevel=2,
        shuffle=True,
        fill_value=FILL_VALUE,
        chunksizes=(
            31,
            len(mask_lat),
            len(mask_lon)
        )
    )


    # -------------------------
    # ATTRIBUT
    # -------------------------

    time_var.units = (
        "days since 1970-01-01 00:00:00"
    )

    time_var.calendar = (
        "proleptic_gregorian"
    )

    time_var.standard_name = "time"

    lat_var[:] = mask_lat.astype(
        "float32"
    )

    lon_var[:] = mask_lon.astype(
        "float32"
    )

    lat_var.units = "degrees_north"
    lon_var.units = "degrees_east"

    utci_var.units = "degC"

    utci_var.long_name = (
        "daily maximum Universal Thermal Climate Index"
    )

    utci_var.daily_reduction = (
        "maximum of 24 hourly UTC values"
    )

    utci_var.source_units = (
        "Kelvin in input files; converted by subtracting 273.15"
        if INPUT_IS_KELVIN
        else "degC"
    )

    out.title = (
        "Daily Maximum UTCI over Sulawesi"
    )

    out.period = (
        f"{START_YEAR}-{END_YEAR}"
    )

    out.spatial_mask = (
        "same Sulawesi native-grid mask as Tmax processing"
    )

    out.temporal_basis = (
        "UTC day: 00:00–23:00 UTC"
    )


    # -------------------------
    # LOOP HARIAN
    # -------------------------

    daily_records = []

    for i, path in enumerate(
        input_files
    ):

        with xr.open_dataset(
            path,
            decode_times=True,
            decode_timedelta=False
        ) as ds:

            if "utci" not in ds.data_vars:
                raise ValueError(
                    f"{path.name}: utci tidak ditemukan."
                )

            da = ds["utci"]

            rename_map = {}

            if "lat" in da.coords:
                rename_map[
                    "lat"
                ] = "latitude"

            if "lon" in da.coords:
                rename_map[
                    "lon"
                ] = "longitude"

            if rename_map:
                da = da.rename(
                    rename_map
                )

            da = da.transpose(
                "time",
                "latitude",
                "longitude"
            )

            times = pd.DatetimeIndex(
                da["time"].values
            )

            if len(times) != 24:
                raise ValueError(
                    f"{path.name}: timestep = {len(times)}, "
                    "seharusnya 24."
                )

            unique_dates = (
                times.normalize().unique()
            )

            if len(unique_dates) != 1:
                raise ValueError(
                    f"{path.name}: berisi lebih dari satu tanggal."
                )

            date = pd.Timestamp(
                unique_dates[0]
            )

            da_crop = da.isel(
                latitude=lat_idx,
                longitude=lon_idx
            )

            values = np.asarray(
                da_crop.max(
                    dim="time",
                    skipna=True
                ).values,
                dtype="float32"
            )

            if INPUT_IS_KELVIN:
                values -= np.float32(
                    273.15
                )

            valid_before_mask = int(
                np.isfinite(values).sum()
            )

            values[
                ~mask_crop
            ] = FILL_VALUE

            day_min = float(
                np.min(
                    values[
                        mask_crop
                    ]
                )
            )

            day_max = float(
                np.max(
                    values[
                        mask_crop
                    ]
                )
            )


        # -------------------------
        # TULIS TIME
        # -------------------------

        numeric_time = (
            (
                date
                - pd.Timestamp(
                    "1970-01-01"
                )
            )
            / pd.Timedelta(
                days=1
            )
        )

        time_var[i] = float(
            numeric_time
        )

        utci_var[
            i,
            :,
            :
        ] = values


        daily_records.append(
            {
                "date": date,
                "file": path.name,
                "daily_min_C": day_min,
                "daily_max_C": day_max,
                "valid_grid_before_mask": valid_before_mask,
                "valid_sulawesi_grid": int(mask_crop.sum()),
            }
        )


        if (
            i < 5
            or (i + 1) % 500 == 0
            or (i + 1) == len(input_files)
        ):

            print(
                f"[{i + 1:05d}/{len(input_files):05d}] "
                f"{date.date()} | "
                f"{day_min:.2f}–{day_max:.2f} °C"
            )


        del values

        if (
            (i + 1) % 100 == 0
        ):
            gc.collect()


daily_qc_df = pd.DataFrame(
    daily_records
)

daily_qc_df.to_csv(
    QC_OUTPUT_FILE,
    index=False
)

print("\nOUTPUT DAILY MAX:")
print(OUTPUT_FILE)

print("\nQC DAILY MAX:")
print(QC_OUTPUT_FILE)

[00001/10958] 1996-01-01 | 26.40–38.31 °C
[00002/10958] 1996-01-02 | 26.33–38.36 °C
[00003/10958] 1996-01-03 | 26.93–37.97 °C
[00004/10958] 1996-01-04 | 26.31–37.26 °C
[00005/10958] 1996-01-05 | 24.84–36.62 °C
[00500/10958] 1997-05-14 | 27.31–37.70 °C
[01000/10958] 1998-09-26 | 28.82–37.96 °C
[01500/10958] 2000-02-08 | 25.54–38.84 °C
[02000/10958] 2001-06-22 | 24.98–36.99 °C
[02500/10958] 2002-11-04 | 27.44–40.45 °C
[03000/10958] 2004-03-18 | 22.96–37.48 °C
[03500/10958] 2005-07-31 | 25.05–38.92 °C
[04000/10958] 2006-12-13 | 27.28–41.61 °C
[04500/10958] 2008-04-26 | 24.49–38.15 °C
[05000/10958] 2009-09-08 | 26.09–39.24 °C
[05500/10958] 2011-01-21 | 28.54–38.60 °C
[06000/10958] 2012-06-04 | 27.88–37.48 °C
[06500/10958] 2013-10-17 | 27.55–42.01 °C
[07000/10958] 2015-03-01 | 28.76–38.62 °C
[07500/10958] 2016-07-13 | 26.50–37.73 °C
[08000/10958] 2017-11-25 | 27.00–40.00 °C
[08500/10958] 2019-04-09 | 30.28–40.68 °C
[09000/10958] 2020-08-21 | 25.73–39.22 °C
[09500/10958] 2022-01-03 | 28.05–3

## 7. QC hasil akhir

Target:
- 10.958 hari;
- 1996-01-01 sampai 2025-12-31;
- tidak ada tanggal duplikat/hilang;
- jumlah grid mask sama dengan hasil Tmax;
- output UTCI sudah dalam °C.

In [8]:
# =========================================================
# 7. QC OUTPUT NETCDF
# =========================================================

with xr.open_dataset(
    OUTPUT_FILE,
    engine="netcdf4",
    decode_times=True,
    decode_timedelta=False
) as out_ds:

    dates = pd.DatetimeIndex(
        out_ds["time"].values
    )

    actual_days = out_ds.sizes[
        "time"
    ]

    actual_lat = out_ds.sizes[
        "latitude"
    ]

    actual_lon = out_ds.sizes[
        "longitude"
    ]

    overall_min = float(
        out_ds["utci_max"]
        .min(
            skipna=True
        )
        .values
    )

    overall_max = float(
        out_ds["utci_max"]
        .max(
            skipna=True
        )
        .values
    )


expected_dates = pd.date_range(
    f"{START_YEAR}-01-01",
    f"{END_YEAR}-12-31",
    freq="D"
)

duplicate_dates = dates[
    dates.duplicated()
]

missing_dates = expected_dates.difference(
    dates
)

extra_dates = dates.difference(
    expected_dates
)


print(
    "========== QC UTCI DAILY MAX =========="
)

print(
    "Jumlah hari aktual  :",
    actual_days
)

print(
    "Jumlah hari harapan :",
    len(expected_dates)
)

print(
    "Tanggal awal        :",
    dates.min()
)

print(
    "Tanggal akhir       :",
    dates.max()
)

print(
    "Tanggal duplikat    :",
    len(duplicate_dates)
)

print(
    "Tanggal hilang      :",
    len(missing_dates)
)

print(
    "Tanggal ekstra      :",
    len(extra_dates)
)

print(
    "Grid output         :",
    (actual_lat, actual_lon)
)

print(
    "Grid Sulawesi mask  :",
    int(mask_crop.sum())
)

print(
    "UTCI minimum        :",
    overall_min,
    "°C"
)

print(
    "UTCI maximum        :",
    overall_max,
    "°C"
)


checks = {
    "jumlah_hari": (
        actual_days
        == len(expected_dates)
    ),

    "duplikat_0": (
        len(duplicate_dates)
        == 0
    ),

    "missing_0": (
        len(missing_dates)
        == 0
    ),

    "extra_0": (
        len(extra_dates)
        == 0
    ),

    "grid_shape": (
        (actual_lat, actual_lon)
        == mask_crop.shape
    ),
}


print(
    "\n========== STATUS STEP 2 =========="
)

for name, status in checks.items():

    print(
        f"{name:15s}:",
        "LULUS"
        if status
        else "GAGAL"
    )


if all(
    checks.values()
):

    print(
        "\nUTCI DAILY MAX + MASK SULAWESI: LULUS"
    )

    print(
        "Data siap masuk STEP 3: "
        "klasifikasi thermal stress UTCI."
    )

else:

    raise RuntimeError(
        "QC STEP 2 belum lulus. "
        "Jangan lanjut ke klasifikasi thermal stress."
    )

========== QC UTCI DAILY MAX ==========
Jumlah hari aktual  : 10958
Jumlah hari harapan : 10958
Tanggal awal        : 1996-01-01 00:00:00
Tanggal akhir       : 2025-12-31 00:00:00
Tanggal duplikat    : 0
Tanggal hilang      : 0
Tanggal ekstra      : 0
Grid output         : (44, 32)
Grid Sulawesi mask  : 242
UTCI minimum        : 18.56707763671875 °C
UTCI maximum        : 44.473175048828125 °C

========== STATUS STEP 2 ==========
jumlah_hari    : LULUS
duplikat_0     : LULUS
missing_0      : LULUS
extra_0        : LULUS
grid_shape     : LULUS

UTCI DAILY MAX + MASK SULAWESI: LULUS
Data siap masuk STEP 3: klasifikasi thermal stress UTCI.


## Output STEP 2

Jika status akhir **LULUS**, file utama yang dipakai pada tahap berikutnya adalah:

`D:\MERLI\data\UTCI_Hasil\03_utci_dailymax_1996_2025_C_sulawesi.nc`

Variabel:
- `utci_max`
- unit: `degC`
- temporal: daily maximum dari 24 jam UTC
- spasial: mask Sulawesi yang sama dengan Tmax

Tahap berikutnya adalah **STEP 3 — klasifikasi UTCI thermal stress dan statistik tahunan**.